In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"data/raw/raw.csv")
df

In [ ]:
df.info()

In [ ]:
cat_columns = [ 'type_of_meal_plan','required_car_parking_space', 'room_type_reserved', 'market_segment_type',
       'repeated_guest','booking_status']
num_columns = ['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights','lead_time',
       'arrival_year', 'arrival_month', 'arrival_date','no_of_previous_cancellations',
       'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests']

##### Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
mappings = {}

for col in cat_columns:
    df[col] = label_encoder.fit_transform(df[col])

    mappings[col] = {label:code for label, code in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))}

In [ ]:
mappings

In [ ]:
df.head()

In [ ]:
df = df.drop("Booking_ID", axis=1)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

##### Multicollineraity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [ ]:
X = add_constant(df)
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values,i) for i in range(X.shape[1])]

In [ ]:
vif_data

In [ ]:
corr = df.corr()
corr

In [ ]:
plt.figure(figsize=(20,15))
sns.heatmap(corr, annot=True, linewidths=0.5)

##### Skewness

In [ ]:
skewness = df.skew()
skewness

In [ ]:
for col in df.columns:
    if skewness[col] > 5:
        df[col] = np.log1p(df[col])

##### Imbalanced Data

In [ ]:
df["booking_status"].value_counts()

In [ ]:
X = df.drop(columns='booking_status')
y = df['booking_status']

In [ ]:
X.columns

In [ ]:
y.value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X, y)

In [ ]:
y_res

In [ ]:
y_res.value_counts( )

In [ ]:
balanced_df = pd.DataFrame(X_res, columns=X.columns)
balanced_df['booking_status'] = y_res

In [ ]:
balanced_df 

In [ ]:
if os.path.exists(r"data\raw\preprocessed_data.csv"):
    print("Preprocessed File already exists")
else:
    balanced_df.to_csv(r"data\raw\preprocessed_data.csv", index=False)